# ImplementForge -- FreshCart Under Pressure
## COSE 278: Implementing Systems -- Day 4

**Run cells in order. Do not skip cells.**
Each phase shows incident cards as interactive widgets.
Pick your decision from the dropdown, explain your rationale, submit.
Then run the consequence cell below.


## Setup
*Run first. Do not modify.*


In [ ]:
GITHUB_BASE = "https://raw.githubusercontent.com/rickghome/278game/main"
GITHUB_REPO = "rickghome/278game"
import urllib.request, os, pickle, json as _json

# Load files
_files_ok = True
for filename in ["if_engine.py", "if_cards.py", "if_widgets.py"]:
    try:
        urllib.request.urlretrieve(f"{GITHUB_BASE}/{filename}", filename)
        print(f"  loaded: {filename}")
    except Exception as e:
        print(f"  FAILED: {filename} -- {e}")
        _files_ok = False

if not _files_ok:
    print()
    print("ERROR: Could not load one or more files from GitHub.")
    print(f"Check that {GITHUB_REPO} is accessible and try again.")
    raise RuntimeError("Setup failed -- do not proceed.")

exec(open("if_engine.py").read())
exec(open("if_cards.py").read())
exec(open("if_widgets.py").read())

# Print version info from GitHub API
try:
    _api_url = f"https://api.github.com/repos/{GITHUB_REPO}/commits/main"
    _req = urllib.request.Request(_api_url, headers={"Accept": "application/vnd.github.v3+json"})
    with urllib.request.urlopen(_req, timeout=5) as _resp:
        _commit = _json.loads(_resp.read())
    _sha   = _commit["sha"][:7]
    _msg   = _commit["commit"]["message"].splitlines()[0][:60]
    _date  = _commit["commit"]["author"]["date"][:10]
    print()
    print(f"ImplementForge loaded.")
    print(f"  repo:    github.com/{GITHUB_REPO}")
    print(f"  commit:  {_sha}  ({_date})")
    print(f"  message: {_msg}")
except Exception:
    print()
    print("ImplementForge loaded.")
    print(f"  repo:    github.com/{GITHUB_REPO}")
    print(f"  commit:  (version info unavailable)")

print()
print("Run the next cell.")


## Save / Restore
*Run second. Do not modify.*


In [ ]:
def save_game(game):
    with open('game_state.pkl','wb') as f:
        pickle.dump(game, f)
    total = sum(game['income'].values())
    print(f"Saved: {game['team_name']}  |  Income: ${total:,}  |  Trust: {game['trust_score']}/100")

def restore_game():
    if not os.path.exists('game_state.pkl'):
        print("No saved game found. Start from Team Setup.")
        return None
    with open('game_state.pkl','rb') as f:
        game = pickle.load(f)
    total = sum(game['income'].values())
    print(f"Restored: {game['team_name']}  |  Income: ${total:,}  |  Trust: {game['trust_score']}/100")
    print(f"Active seeds: {game.get('seeds',[])}")
    return game

print("Save/restore ready.")


## Team Setup


In [ ]:
TEAM_NAME = "Team Name Here"   # <- change this

game = new_game_state(TEAM_NAME)
print(f"Team registered: {TEAM_NAME}")
print(f"Starting trust score: {game['trust_score']}/100")
print(f"Environment: set in Frame 1")


## Frame 1 -- Architecture Config

You designed FreshCart in COSE 260. Now you are building it.
Fill in each field and run the cell.
**Read the comments -- they are your options.**


In [ ]:
system_profile = {

    "environment": "consumer",
    # consumer    -- fast market, high volume, volatile trust
    # enterprise  -- contract-driven, stable, reputational risk
    # government  -- fixed budget, procurement rules, political exposure

    "team_structure": "stream_aligned",
    # stream_aligned  -- small teams, end-to-end ownership, low coordination cost
    # platform        -- shared services team supporting others
    # siloed          -- functional departments, high handoff cost

    "build_buy_configure": "build",
    # build       -- we write it ourselves
    # buy         -- we purchase a product
    # configure   -- we implement a vendor platform

    "primary_risk": "delivery_speed",
    # delivery_speed   -- we might be too slow
    # technical_debt   -- we might cut corners
    # integration      -- our components might not fit together
    # vendor_lock      -- we might become too dependent on third parties
    # team_capability  -- we might not have the right skills

    "data_architecture": "shared_db",
    # dedicated_dbs  -- each service owns its data -- higher cost, clean boundaries
    # shared_db      -- single shared database -- lower cost, faster to build

    "architecture_pattern": "monolith",
    # monolith      -- single deployable unit -- simple, fast, coupling debt deferred
    # layered       -- structured layers -- some overhead, coupling risk over time
    # client_server -- central server -- familiar, bottleneck risk at scale
    # event_driven  -- async messaging -- scalable, hard to debug without tooling
    # microservices -- independent services -- flexible, high coordination cost upfront

    "coupling": "medium",   # fixed -- do not change
}

is_valid, errors, warnings = validate_frame1(system_profile)

if errors:
    print("Fix these errors before continuing:")
    for e in errors:
        print(f"  {e}")
else:
    game["frame1"]     = system_profile
    game["environment"] = system_profile["environment"]
    print("Frame 1 accepted.")
    print()
    for w in warnings:
        print(w)
    print()
    print(f"Environment:      {system_profile['environment']}")
    print(f"Team structure:   {system_profile['team_structure']}")
    print(f"Strategy:         {system_profile['build_buy_configure']}")
    print(f"Primary risk:     {system_profile['primary_risk']}")
    print(f"Data:             {system_profile['data_architecture']}")
    print(f"Architecture:     {system_profile['architecture_pattern']}")
    apply_architecture_tax(game)
    save_game(game)


## Architecture Stress Test

Your architecture choice just met its first real constraint.
One card fires based on your pattern. Only your decision controls the damage.

**Run this cell to see your card. Use the widget to submit your decision.**


In [ ]:
_arch_cards = select_architecture_cards(system_profile, game)
show_phase_cards("Architecture Stress", _arch_cards, game)


In [ ]:
# Consequence cell -- run after submitting your decision above
run_phase_consequences("arch_stress", game)


## Frame 2 -- Pipeline Config

You have seen the Architecture Stress. Now configure your pipeline.

**Engineering Capacity hard cap: 100 units.**
The mature stack costs 95. The minimal stack costs 10. Neither is free.


In [ ]:
# Capacity costs (for reference while configuring):
# testing:      minimal=5   standard=15  thorough=30
# deployment:   big_bang=5  rolling=15   blue_green=25  canary=30
# rollback:     none=0      partial=10   full=20
# observability:none=0      basic=10     full=25
# change_owner: single=0    pair=10      team=20

pipeline_profile = {
    "testing_coverage":    "standard",
    # minimal | standard | thorough

    "deployment_method":   "rolling",
    # big_bang | rolling | blue_green | canary

    "rollback_plan":       "partial",
    # none | partial | full

    "observability_level": "basic",
    # none | basic | full

    "change_owner":        "shared_pair",
    # single_person | shared_pair | team_owned
}

is_valid, errors, warnings, capacity_used = validate_frame2(pipeline_profile)
print(f"Engineering Capacity used: {capacity_used}/100 units")
print()

if errors:
    print("Fix these errors before continuing:")
    for e in errors:
        print(f"  {e}")
else:
    game["frame2"] = pipeline_profile
    print("Frame 2 accepted.")
    for w in warnings:
        print(w)
    print()
    print(f"Testing:        {pipeline_profile['testing_coverage']}")
    print(f"Deployment:     {pipeline_profile['deployment_method']}")
    print(f"Rollback:       {pipeline_profile['rollback_plan']}")
    print(f"Observability:  {pipeline_profile['observability_level']}")
    print(f"Change owner:   {pipeline_profile['change_owner']}")
    save_game(game)


## Build Phase

FreshCart is under construction. Org and architecture choices are meeting reality.

**Run this cell to see your cards. Use the widgets to submit decisions.**


In [ ]:
_build_cards = select_build_cards(system_profile, game)
show_phase_cards("Build", _build_cards, game)


In [ ]:
# Consequence cell -- run after submitting all decisions above
run_phase_consequences("build", game)


## Frame 3 -- Operations Config

You have seen Architecture Stress and Build. Configure your operational layer.


In [ ]:
operations_profile = {
    "vendor_dependency":  "medium",
    # low | medium | high

    "fallback_strategy":  "manual",
    # none | manual | automated

    "on_call_coverage":   "business_hours",
    # none | business_hours | follow_the_sun | full_24x7

    "incident_response":  "runbook",
    # ad_hoc | runbook | practiced
}

is_valid, errors, warnings = validate_frame3(operations_profile)

if errors:
    print("Fix these errors before continuing:")
    for e in errors:
        print(f"  {e}")
else:
    game["frame3"] = operations_profile
    print("Frame 3 accepted.")
    for w in warnings:
        print(w)
    print()
    print(f"Vendor dependency:  {operations_profile['vendor_dependency']}")
    print(f"Fallback strategy:  {operations_profile['fallback_strategy']}")
    print(f"On-call coverage:   {operations_profile['on_call_coverage']}")
    print(f"Incident response:  {operations_profile['incident_response']}")
    save_game(game)


## Pipeline Gates

Your change moves from development toward production through five gates.

**What you skip here becomes a production incident later.**
The cost curve is real: a bug caught in unit test costs $8k average.
The same bug in production costs $64k average.

Gates 1-4 are investment decisions. Gate 5 is your go/no-go.
Fill in the config and run the cell.


In [ ]:
pipeline_gates = {

    # GATE 1 -- UNIT TEST
    # Who owns it: Developer
    # Question: does each piece work as designed?
    "unit_test_strategy": "full",
    # full        -- all code paths including edge cases
    # happy_path  -- main flows only, skip edge cases
    # skip        -- no unit tests run

    # GATE 2 -- INTEGRATION TEST
    # Who owns it: Dev + QA jointly
    # Question: do the pieces work together?
    "integration_owner": "dev_and_qa",
    # dev_and_qa  -- joint ownership, catches cross-team assumptions
    # dev_only    -- fast, misses what QA would catch
    # skip        -- nothing tested together

    "integration_scope": "full",
    # full          -- all integration points tested
    # critical_path -- main flows only
    # skip          -- not tested

    # GATE 3 -- STAGING
    # Who owns it: QA + Ops
    # Question: does it work in a production-like environment?
    "staging_fidelity": "production_mirror",
    # production_mirror -- same data volume, config, network topology as production
    # partial_mirror    -- similar config, but smaller data volume
    # dev_extended      -- staging is just dev with a different name

    # GATE 4 -- UAT
    # Who owns it: BUSINESS USERS -- not IT
    # Question: does it do what the business actually needs?
    "uat_owner": "business_users",
    # business_users -- real users, real workflows, real questions
    # it_team        -- faster, but answers the wrong question
    # skip           -- no user validation before go-live

    "uat_coverage": "full_workflows",
    # full_workflows -- all business processes tested end to end
    # happy_path     -- main journeys only
    # skip           -- no coverage

    # GATE 5 -- GO / NO-GO
    # Based on what the pipeline found, do you ship?
    "go_nogo_decision": "go",
    # go              -- ship as planned
    # conditional_go  -- ship with documented known issues
    # no_go           -- delay and fix first
}

is_valid, errors, warnings = validate_pipeline_gates(pipeline_gates)

if errors:
    print("Fix these errors before continuing:")
    for e in errors:
        print(f"  {e}")
else:
    game["pipeline_gates"] = pipeline_gates
    print("Pipeline gates accepted.")
    print()
    for w in warnings:
        print(w)
    print()

    # Calculate costs, seeds, bug catch rate
    total_cost, gate_seeds, bugs_caught, cost_breakdown = calculate_pipeline_costs(
        pipeline_gates, game["environment"]
    )

    # Plant gate seeds into game state
    for s in gate_seeds:
        plant_seed(game, s)

    # Deduct pipeline cost from phase income
    env = game["environment"]
    vel = game.get("velocity_multiplier", 1.0)
    phase_income = int(_baseline(env) * vel) - total_cost
    update_income(game, "pipeline", max(0, phase_income))

    # Print cost curve and go/no-go assessment
    print_pipeline_cost_curve(pipeline_gates, cost_breakdown, bugs_caught, gate_seeds, env)
    print_go_nogo_assessment(pipeline_gates, gate_seeds, game)

    save_game(game)


## Live Phase

FreshCart is live. Real load. Real users. Real consequences.

One card in this phase has a time limit -- decide quickly.

**Run this cell to see your cards. Use the widgets to submit decisions.**


In [ ]:
_live_cards = select_live_cards(system_profile, pipeline_profile, game)
show_phase_cards("Live", _live_cards, game)


In [ ]:
# Consequence cell -- run after submitting all decisions above
run_phase_consequences("live", game)


## v2 Release

**Universal requirement -- same for every team:**

> FreshCart is adding a real-time delivery tracking feature.
> New Tracking Service, updated mobile API contract, change to Notification Service.
> All teams implement this. What happens next depends on what you built.

**Run this cell to see your cards. Use the widgets to submit decisions.**


In [ ]:
_v2_cards  = select_v2_cards(system_profile, pipeline_profile, game)
_seed_count = count_seeds(game)

if _seed_count >= 2:
    print(f"Entering v2 Release with {_seed_count} active seeds. Severity escalated on first card.")
    print()

show_phase_cards("v2 Release", _v2_cards, game)


In [ ]:
# Consequence cell -- run after submitting all decisions above
run_phase_consequences("v2_release", game)


## Scale Event

**Wait for your instructor to read the scenario aloud.**

No new config. No new decisions on this one.
Your architecture holds -- or it doesn't.
Run the cell when instructed.


In [ ]:
def _calculate_scale_outcome(game):
    env   = game["environment"]
    idx   = ENV_IDX[env]
    base  = int(PHASE_BASELINE[idx] * game.get("velocity_multiplier", 1.0))
    seeds = game.get("seeds", [])
    f2    = game.get("frame2", {})
    f3    = game.get("frame3", {})

    # Resilience score from config choices
    score = 0
    deploy = f2.get("deployment_method", "big_bang")
    if deploy in ["canary", "blue_green"]: score += 2
    elif deploy == "rolling":              score += 1

    rollback = f2.get("rollback_plan", "none")
    if rollback == "full":     score += 2
    elif rollback == "partial": score += 1

    obs = f2.get("observability_level", "none")
    if obs == "full":    score += 2
    elif obs == "basic": score += 1

    fallback = f3.get("fallback_strategy", "none") if f3 else "none"
    if fallback == "automated": score += 2
    elif fallback == "manual":  score += 1

    ir = f3.get("incident_response", "ad_hoc") if f3 else "ad_hoc"
    if ir == "practiced": score += 2
    elif ir == "runbook":  score += 1

    effective = score - len(seeds) * 0.5
    is_gov    = env == "government"

    if effective >= 7:
        tier, income, td = "THRIVING", base + int(base * 0.68), 10
        narrative = (
            "Traffic spiked 35x. Systems flagged it in 90 seconds.\n"
            "Automated fallback activated. FreshCart degraded gracefully.\n\n"
            f"Revenue captured: +${int(base*0.68):,} above baseline.\n"
            "Trust: +10. This is what you paid for."
        )
    elif effective >= 4:
        tier, income, td = "SURVIVING", int(base * 0.95), 0
        narrative = "Traffic spiked 35x. FreshCart bent but did not break. Most revenue captured."
    elif effective >= 1:
        tier, income, td = "STRUGGLING", int(base * 0.6), -15
        narrative = "Traffic spiked 35x. FreshCart struggled. Major revenue loss. Trust eroded."
    else:
        tier, income, td = "COLLAPSING", int(base * 0.2), -30
        narrative = (
            "Traffic spiked 35x. FreshCart collapsed under load.\n"
            "The accumulated decisions of the past phases produced this outcome."
        )

    if is_gov:
        income = max(income, int(base * 0.4))

    # Resolve planted seeds
    seed_map = {
        "P1_silent_fraud":     ((400_000, 300_000, 240_000), -40, "P1 silent fraud surfaces publicly."),
        "V2_mock_data":        ((200_000, 150_000, 120_000), -35, "V2 mock data discovered."),
        "D1_db_scaling":       ((150_000, 112_000,  90_000), -10, "DB saturation triggered."),
        "P3_workaround_live":  ((120_000,  90_000,  72_000), -10, "P3 config cascade."),
        "V1_queue_risk":       ((180_000, 135_000, 108_000), -25, "V1 notification queue overflow."),
        "P2_shared_ownership": ((100_000,  75_000,  60_000), -15, "P2 ownership failure repeats."),
        "PT1_integration_debt":((80_000,   60_000,  48_000), -10, "Integration debt surfaces."),
        "PT2_env_mismatch":    ((60_000,   45_000,  36_000),  -5, "Environment mismatch in production."),
        "PT3_uat_skip":        ((90_000,   67_000,  54_000), -15, "UAT bug found by customers."),
        "PT4_race_condition":  ((70_000,   52_000,  42_000), -10, "Race condition triggered at scale."),
        "PT5_prod_config_risk":((80_000,   60_000,  48_000),  -8, "Production config failure."),
        "A1_coupling_debt":    ((120_000,  90_000,  72_000), -10, "Monolith coupling collapses under load."),
        "A2_layer_debt":       ((80_000,   60_000,  48_000),  -8, "Layer violations compound."),
        "A3_scale_cliff":      ((200_000, 150_000, 120_000), -20, "Client-server hits scale cliff."),
        "A4_trace_gap":        ((90_000,   67_000,  54_000), -10, "Async failures untraceable."),
        "A5_stakeholder_doubt":((60_000,   45_000,  36_000), -15, "Stakeholder confidence lost."),
        "gate_unit_skipped":   ((80_000,   60_000,  48_000), -10, "Undetected unit-level bugs surface."),
        "gate_integration_skipped":((100_000,75_000, 60_000),-12, "Integration failures hit production."),
        "gate_staging_fake":   ((90_000,   67_000,  54_000), -10, "Staging did not catch production issues."),
        "gate_uat_wrong_owner":((80_000,   60_000,  48_000), -15, "Business workflows broken post-launch."),
        "gate_uat_skipped":    ((120_000,  90_000,  72_000), -20, "No UAT -- business found the gaps."),
        "gate_red_light_go":   ((150_000, 112_000,  90_000), -20, "Shipped on RED pipeline assessment."),
    }

    seed_notes, extra_loss, extra_trust = [], 0, 0
    for seed_id, (losses, trust_hit, note) in seed_map.items():
        if seed_id in seeds:
            extra_loss  += losses[idx]
            extra_trust += trust_hit
            seed_notes.append(note)

    income  = max(0, income - extra_loss)
    td     += extra_trust
    if is_gov:
        income = max(income, int(base * 0.4))

    return tier, income, td, narrative, seed_notes

# Run scale event
tier, s_income, s_trust, narrative, seed_notes = _calculate_scale_outcome(game)
game["cards_fired"].append("S1")

print("-" * 50)
print("SCALE EVENT -- THE MOMENT OF TRUTH")
print("-" * 50)
print()
print(narrative)
print()

if seed_notes:
    print("Seeds resolving now:")
    for note in seed_notes:
        print(f"  -> {note}")
    print()

update_income(game, "scale", s_income)
update_trust(game, s_trust)
print(f"Outcome: {tier}")
print()
print_income_summary(game)
save_game(game)


## The Bill Arrives

Trust score is revealed first. The room sits with it.
Then revenue impact. Then your decision.


In [ ]:
print("-" * 50)
print("THE BILL ARRIVES")
print("-" * 50)
print()
print(f"Your Trust Score: {game['trust_score']}/100")
env_label = {
    "consumer":   "Customer Trust",
    "enterprise": "Stakeholder Confidence",
    "government": "Political Capital",
}.get(game["environment"], "Trust")
print(f"({env_label})")
print()
print("... room discussion ...")
print()
print("Run the next cell to reveal revenue impact and enter your decision.")


In [ ]:
# Show S2 as widget -- consequence cell handles card tracking
show_phase_cards("The Bill Arrives", ["S2"], game)


In [ ]:
# Consequence cell for S2 -- separate processed flag from S1 scale event
run_phase_consequences("bill_arrives", game)


## Political Exposure
*Government environment only. Skip if not government.*


In [ ]:
if game.get("frame1", {}).get("environment") != "government":
    print("Not government environment. Skip to Post-Mortem.")
elif not _should_fire_S3(game.get("frame1", {}), game):
    print("S3 did not fire for this team.")
else:
    show_phase_cards("Political Exposure", ["S3"], game)


In [ ]:
if game.get("frame1", {}).get("environment") == "government" and _should_fire_S3(game.get("frame1",{}), game):
    run_phase_consequences("political_exposure", game)
else:
    print("Skip.")


## Final Summary
*Copy the output and give it to your instructor.*


In [ ]:
print_final_summary(game)


## Post-Mortem

The game is over. Now the learning begins.

This is not blame. This is the system finding its own failure mode.
Fill in all three fields and run the cell.


In [ ]:
postmortem = {

    "assumption_that_failed": "speed_over_quality",
    # speed_over_quality          -- we optimized for delivery speed over quality
    # team_knows_the_system       -- tribal knowledge was sufficient
    # vendor_will_deliver         -- we trusted the vendor roadmap
    # staging_mirrors_prod        -- staging was a good enough proxy for production
    # users_will_adapt            -- users would figure out the gaps
    # debt_can_wait               -- we could pay down debt later
    # architecture_will_scale     -- our architecture would hold under real load
    # pipeline_gates_are_optional -- gates were bureaucracy, not protection

    "earliest_catch_point": "unit_test_gate",
    # architecture_decision    -- Frame 1 config choice
    # team_structure_choice    -- org design
    # unit_test_gate           -- Gate 1
    # integration_gate         -- Gate 2
    # staging_gate             -- Gate 3
    # uat_gate                 -- Gate 4
    # go_nogo_decision         -- Gate 5
    # live_incident            -- first production failure
    # v2_release               -- v2 compounding

    "one_thing_different": "Describe one concrete change you would make before starting again",
}

if postmortem["one_thing_different"].startswith("Describe"):
    print("Fill out all three fields before running.")
else:
    game["postmortem"] = postmortem
    print_postmortem_analysis(postmortem, game)
    save_game(game)


## Reflection -- Before Second Iteration

Fill this out honestly before you change anything.


In [ ]:
strategy_change = {
    "what_failed":    "Describe the main thing that went wrong",
    "what_we_change": "Describe the specific config change you are making",
    "why":            "One sentence: why do you believe this change fixes the problem",
}

if any(v.startswith("Describe") for v in strategy_change.values()):
    print("Fill out all three fields before continuing.")
else:
    game["strategy_change"] = strategy_change
    print("Reflection recorded:")
    for k, v in strategy_change.items():
        print(f"  {k}: {v}")
    save_game(game)
    print()
    print("Run the next cell to begin the second iteration.")


## Second Iteration Setup


In [ ]:
import copy

if not game.get("strategy_change") or any(
    v.startswith("Describe") for v in game["strategy_change"].values()
):
    print("Complete the reflection first.")
else:
    # Save iteration 1 state for comparison
    game_v1 = copy.deepcopy(game)

    # Reset for iteration 2
    game["income"] = {
        "arch_stress": 0,
        "build":       0,
        "pipeline":    0,
        "live":        0,
        "v2_release":  0,
        "scale":       0,
    }
    game["trust_score"]       = 100
    game["cards_fired"]       = []
    game["decisions"]         = {}
    game["rationales"]        = {}
    game["facilitator_trace"] = []
    game["seeds"]             = []
    game["phase"]             = 0
    game["iteration"]         = 2
    game["pending_decisions"] = {}

    # Clear phase-processed flags
    for key in list(game.keys()):
        if key.endswith("_processed"):
            del game[key]

    print("Second iteration ready.")
    print("Scroll back to Frame 1, update your config, then run all cells again.")
    print()
    print("At the end, the engine will compare both iterations.")


## Iteration Comparison
*Run after completing both iterations.*


In [ ]:
try:
    print_iteration_comparison(game_v1, game)
except NameError:
    print("Complete both iterations before comparing.")
